# Objective C (Exploratory Probe): Inflation Targeting Event Study

## Caveats (read first)

1. IT adoption timing can be endogenous to prior inflation performance.
2. Roger (2010) and Hammond (2012) adoption dates are not identical for all countries.
3. A DiD on inflation levels identifies a level shift; slope-shift interpretation requires explicit interaction design.
4. This notebook is an exploratory probe, not a final causal policy evaluation.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from linearmodels.panel import PanelOLS


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "02_data/analysis_ready/macro_growth_merged.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find project root with analysis-ready data.")


ROOT = find_project_root(Path.cwd().resolve())
PANEL_PATH = ROOT / "02_data/analysis_ready/macro_growth_merged.csv"
IT_DATES_PATH = ROOT / "02_data/supporting/it_adoption_dates.csv"
OUT_DIR = ROOT / "03_analysis_notebooks/exports/objective_c"
OUT_DIR.mkdir(parents=True, exist_ok=True)

panel = pd.read_csv(PANEL_PATH).rename(columns={"Country Name": "country"})
it = pd.read_csv(IT_DATES_PATH)

df = panel.merge(it, on="country", how="left")
df["it_treated"] = df["it_adoption_year_roger2010"].notna().astype(int)
df["post_it"] = (df["year"] >= df["it_adoption_year_roger2010"]).fillna(False).astype(int)
df["event_time"] = df["year"] - df["it_adoption_year_roger2010"]

df[["country", "year", "inflation", "m2_growth", "it_treated", "post_it", "event_time"]].head()

,country,year,inflation,m2_growth,it_treated,post_it,event_time
0,Afghanistan,2007,0.083243,0.353436,0,0,NaN
1,Afghanistan,2008,0.234429,0.272953,0,0,NaN
2,Afghanistan,2009,-0.070542,0.285518,0,0,NaN
3,Afghanistan,2010,0.021551,0.238600,0,0,NaN
4,Afghanistan,2011,0.111579,0.193172,0,0,NaN


In [2]:
# TWFE event-study on inflation levels (exploratory)
event = df.copy()
event["event_bin"] = event["event_time"].clip(lower=-5, upper=5)

for k in range(-5, 6):
    if k == -1:
        continue
    event[f"evt_{k}"] = ((event["event_bin"] == k) & (event["it_treated"] == 1)).astype(int)

rhs_terms = " + ".join([f"`evt_{k}`" for k in range(-5, 6) if k != -1])
fit = event[["country", "year", "inflation", *[f"evt_{k}" for k in range(-5, 6) if k != -1]]].dropna()
fit = fit.set_index(["country", "year"])

es_res = PanelOLS.from_formula(
    f"inflation ~ 1 + {rhs_terms} + EntityEffects + TimeEffects",
    data=fit,
).fit(cov_type="clustered", cluster_entity=True)

es_table = pd.DataFrame({
    "term": es_res.params.index,
    "coef": es_res.params.values,
    "p_value": es_res.pvalues.values,
})
es_table.to_csv(OUT_DIR / "objective_c_event_study_levels.csv", index=False)
es_table.head(12)

,term,coef,p_value
0,Intercept,0.080768,0.000000
1,evt_-5,0.128364,0.222198
2,evt_-4,0.049975,0.111405
3,evt_-3,0.028515,0.058128
4,evt_-2,0.009582,0.247125
5,evt_0,-0.007388,0.333180
6,evt_1,-0.026299,0.050812
7,evt_2,-0.021593,0.228626
8,evt_3,-0.023023,0.203220
9,evt_4,-0.010051,0.569048


In [3]:
# Slope-focused probe: proper triple-diff (corrected D-1)
# post_it_m2 = post_it * m2_growth is collinear with post_treated_m2 for this design:
#   For IT-treated (it_treated=1): post_it_m2 == post_treated_m2 (perfect collinearity)
#   For never-adopters (it_treated=0): post_it == 0, so post_it_m2 == 0 always
# The correct full-rank specification uses only two lower-order interactions:
#   it_m2 = it_treated * m2_growth  (pre-adoption slope diff: treated vs control)
#   post_treated_m2 = post_it * it_treated * m2_growth  (post-adoption slope change)

slope_df = df[["country", "year", "inflation", "m2_growth", "it_treated", "post_it"]].dropna().copy()
slope_df["it_m2"]           = slope_df["it_treated"] * slope_df["m2_growth"]
slope_df["post_treated_m2"] = slope_df["post_it"] * slope_df["it_treated"] * slope_df["m2_growth"]
slope_df = slope_df.set_index(["country", "year"])

slope_res = PanelOLS.from_formula(
    "inflation ~ 1 + m2_growth + it_m2 + post_treated_m2 + EntityEffects + TimeEffects",
    data=slope_df,
).fit(cov_type="clustered", cluster_entity=True)

slope_table = pd.DataFrame([
    {
        "coef_m2_growth": float(slope_res.params["m2_growth"]),
        "p_m2_growth": float(slope_res.pvalues["m2_growth"]),
        "coef_it_m2": float(slope_res.params["it_m2"]),
        "p_it_m2": float(slope_res.pvalues["it_m2"]),
        "coef_post_treated_m2": float(slope_res.params["post_treated_m2"]),
        "p_post_treated_m2": float(slope_res.pvalues["post_treated_m2"]),
        "nobs": int(slope_res.nobs),
    }
])

# Roger-vs-Hammond date sensitivity snapshot
sens = df.copy()
sens["post_it_hammond"] = (sens["year"] >= sens["it_adoption_year_hammond2012"]).fillna(False).astype(int)
sensitivity_table = pd.DataFrame([
    {
        "treated_rows_roger": int((sens["post_it"] == 1).sum()),
        "treated_rows_hammond": int((sens["post_it_hammond"] == 1).sum()),
        "difference_rows": int((sens["post_it"] - sens["post_it_hammond"]).abs().sum()),
    }
])

slope_table.to_csv(OUT_DIR / "objective_c_slope_probe.csv", index=False)
sensitivity_table.to_csv(OUT_DIR / "objective_c_date_sensitivity.csv", index=False)

print("D-1 fixed: proper triple-diff with full-rank specification")
print(f"Interpretation: post_treated_m2 = slope change for IT adopters post-adoption")
slope_table


D-1 fixed: proper triple-diff with full-rank specification
Interpretation: post_treated_m2 = slope change for IT adopters post-adoption


,coef_m2_growth,p_m2_growth,coef_it_m2,p_it_m2,coef_post_treated_m2,p_post_treated_m2,nobs
0,0.635978,0.001586,0.211944,0.281496,-0.484642,0.000004,4750
